In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [ ]:
pip install transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# EDA

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

In [ ]:
train.head()

In [ ]:
train['answer'].value_counts()

In [ ]:
train.isnull().sum()

# Evaluation Metric

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    predictions = np.argsort(logits, axis=-1)[:, ::-1][:, :3]
    
    score = 0.0
    for actual, pred in zip(labels, predictions):
        if actual == pred[0]:
            score += 1.0
        elif actual == pred[1]:
            score += 0.5
        elif actual == pred[2]:
            score += 1/3
            
    return {"map3": score / len(labels)}

# Model 1: LSTM + Transformer

In [ ]:
class HybridQAModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, lstm_hidden=128, num_heads=4, num_layers=2):
        super(HybridQAModel, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, lstm_hidden, batch_first=True, bidirectional=True)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=lstm_hidden * 2, 
            nhead=num_heads, 
            batch_first=True)
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(lstm_hidden * 2, 1)

    def forward(self, input_ids):
        batch_size, num_options, seq_len = input_ids.shape
        x = input_ids.view(batch_size * num_options, seq_len)
        x = self.embedding(x) #
        lstm_out, _ = self.lstm(x) #
        trans_out = self.transformer(lstm_out) 
        pooled = trans_out.mean(dim=1) 
        logits = self.classifier(pooled) 
        logits = logits.view(batch_size, num_options)
        
        return logits

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=128):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        prompt = str(row['prompt'])
        options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
        
        input_ids = []
        for opt in options:
            encoded = self.tokenizer(
                prompt, opt, 
                truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'].squeeze(0))
            
        label = self.label_map[row['answer']]
        
        return torch.stack(input_ids), torch.tensor(label, dtype=torch.long)

In [ ]:
def train_scratch_model(train_loader, val_loader, vocab_size, epochs=3):
    wandb.init(project="smart-mcq-solver", name="Scratch-LSTM-Transformer")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = HybridQAModel(vocab_size=vocab_size).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_inputs, batch_labels in train_loader:
            batch_inputs, batch_labels = batch_inputs.to(device), batch_labels.to(device)
            
            optimizer.zero_grad()
            logits = model(batch_inputs) 
            
            loss = criterion(logits, batch_labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        model.eval()
        val_loss, correct, map3_score = 0, 0, 0
        
        with torch.no_grad():
            for batch_inputs, batch_labels in val_loader:
                batch_inputs, batch_labels = batch_inputs.to(device), batch_labels.to(device)
                logits = model(batch_inputs)
                
                loss = criterion(logits, batch_labels)
                val_loss += loss.item()
                
                preds = torch.argsort(logits, dim=1, descending=True)
                correct += (preds[:, 0] == batch_labels).sum().item()
                
                for i in range(len(batch_labels)):
                    actual = batch_labels[i].item()
                    top3 = preds[i, :3].tolist()
                    if actual in top3:
                        rank = top3.index(actual) + 1
                        map3_score += 1.0 / rank

        avg_train_loss = total_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        val_acc = correct / len(val_loader.dataset)
        val_map3 = map3_score / len(val_loader.dataset)
        
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_accuracy": val_acc,
            "val_map@3": val_map3
        })
        
        print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | MAP@3: {val_map3:.4f}")

    wandb.finish()
    return model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
vocab_size = tokenizer.vocab_size

train_records = train_df.to_dict('records')
val_records = val_df.to_dict('records')
test_records = test.to_dict('records')

train_dataset = MCQDataset(train_records, tokenizer)
val_dataset = MCQDataset(val_records, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

trained_model = train_scratch_model(train_loader, val_loader, vocab_size, epochs=3)

In [ ]:
class TestMCQDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=128):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        prompt = str(row['prompt'])
        options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]

        input_ids = []
        for opt in options:
            encoded = self.tokenizer(
                prompt, opt,
                truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt'
            )
            input_ids.append(encoded['input_ids'].squeeze(0))

        return torch.stack(input_ids), row['id']

test_dataset = TestMCQDataset(test_records, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

trained_model.eval()

In [ ]:
predictions = []
test_ids = []
idx_to_letter = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

with torch.no_grad():
    for batch_inputs, batch_ids in test_loader:
        batch_inputs = batch_inputs.to(device)
        
        logits = trained_model(batch_inputs)
        
        top3_preds = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().numpy()
        
        for i in range(len(batch_ids)):
            letters = [idx_to_letter[idx] for idx in top3_preds[i]]
            predictions.append(" ".join(letters))
            test_ids.append(batch_ids[i].item() if isinstance(batch_ids[i], torch.Tensor) else batch_ids[i])

# Submission Cell

In [ ]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()